### Data preprocessing


In [39]:
import pandas as pd
from sklearn.model_selection import train_test_split
BASE = "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/"
full = pd.read_csv(BASE + "training_data.csv")
test = pd.read_csv(BASE + "testing_data.csv") # scellé jusqu'à la Partie F
full["id"], test["id"] = range(len(full)), range(len(test))

In [40]:
print(full.isnull().sum())

tweet        0
sarcasm      0
sentiment    0
dialect      0
id           0
dtype: int64


### full Dups, tweet Dups

In [41]:
full._dups = full.duplicated()
print(f"Number of duplicate lines in the full set: {full._dups.sum()}")

Number of duplicate lines in the full set: 0


C:\Users\pc\AppData\Local\Temp\ipykernel_21472\175053858.py:1: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  full._dups = full.duplicated()


In [42]:
tweet_dups = full["tweet"].duplicated(keep="first")
print(f"Number of duplicate tweets in the training set: {tweet_dups.sum()}")


Number of duplicate tweets in the training set: 61


In [43]:
dup_tweets = []

def show_dup_tweets(df):
    for index, row in df.iterrows():
        if df.duplicated(subset=["tweet"], keep=False).iloc[index]:
            dup_tweets.append(row)

show_dup_tweets(full)
dup_tweets_df = pd.DataFrame(dup_tweets)
display(dup_tweets_df)

,tweet,sarcasm,sentiment,dialect,id
139,"""RT @mh_awadi: #مصر في الأخبار :أغذية ملوثةأكل...",True,NEG,egypt,139
163,"""RT @A822822A: ياروافض بلا كلام وإلا سوف أخرج♦...",True,NEG,msa,163
541,"""RT @safa_tv: #العبادي يرد على #أردوغان :بالتأ...",False,NEG,msa,541
646,"""RT @BassamJaara: حلب لن تسقط واهلها لن يغادرو...",False,NEG,msa,646
689,"""RT @SYRIA_KIDS: شكراً #بشارآثرتهم على نفسك .و...",False,NEG,msa,689
...,...,...,...,...,...
11872,"""@sawabcenter داعش مجرمين قتلة ولكن النظام الس...",True,NEG,msa,11872
11897,"""RT @HUN_0_TER: @Counselkremlin الرئيس #الروسي...",False,NEU,msa,11897
12158,"""RT @ajmubasher: وزير خارجية #بريطانيا يدعو لل...",False,NEU,msa,12158
12280,"""RT @M__albugaily: هاجموا الحبيبة مصر بضراوة ل...",False,NEG,egypt,12280


In [44]:
full.drop_duplicates(subset=["tweet"], keep=False, inplace=True)

In [45]:
full["tweet"].isnull().sum()

np.int64(0)

### Mapping Emojis in arabic


In [46]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0
1,"""مع فيدرر يا آجا والكبار 😍 https://t.co/hrBeHb...",False,NEU,msa,1
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...",True,NEG,gulf,3
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...",False,NEU,msa,4
5,"""طبيب #المقاصة: إصابة #أحمد_الشيخ بسيطة تحتاج ...",False,NEG,msa,5
6,مرسي : مش هنام اكتر من اربع ساعات في اليوم.......,True,NEG,egypt,6
7,#انتخبوا_البرص مشكلة,True,NEG,egypt,7
8,"""Retweeted هاشتاق العرب (@TheArabHash):#فيديو ...",False,POS,msa,8
9,▪️محافظ العاصمة تفقد سير العملية الانتخابية في...,False,NEU,msa,9


In [47]:
from arabic_emojipedia.emoji_description import get_emoji_description
import emoji

emoji = "😊"

description = get_emoji_description(emoji)

print(f"Description for {emoji}: {description}")

Description for 😊: وجه مبتسم بعيون مبتسمة


In [48]:
import emoji
from arabic_emojipedia.emoji_description import get_emoji_description

def replace_emoji_with_desc(text):

    return emoji.replace_emoji(text, replace=lambda chars,data_dict: f" {get_emoji_description(chars)} ")

full["tweet"] = full["tweet"].apply(replace_emoji_with_desc)

In [49]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0
1,"""مع فيدرر يا آجا والكبار وجه مبتسم بعيون قلب ...",False,NEU,msa,1
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...",True,NEG,gulf,3
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...",False,NEU,msa,4
5,"""طبيب #المقاصة: إصابة #أحمد_الشيخ بسيطة تحتاج ...",False,NEG,msa,5
6,مرسي : مش هنام اكتر من اربع ساعات في اليوم.......,True,NEG,egypt,6
7,#انتخبوا_البرص مشكلة,True,NEG,egypt,7
8,"""Retweeted هاشتاق العرب (@TheArabHash):#فيديو ...",False,POS,msa,8
9,مربع صغير أسود محافظ العاصمة تفقد سير العملية...,False,NEU,msa,9


### Punctuation and special characters removal

Keeping the @ and # actual values as they can be important

In [50]:
import re

def clean_tweet_punctuation(text):
    if not isinstance(text, str):
        return text
    
    # 1. Remove URLs first
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 2. Strip leading and trailing CSV artifact quotes/whitespace
    # (keeps quotes that appear inside the text)
    text = text.strip()
    text = re.sub(r'^["\'“”«»]+|["\'“”«»]+$', '', text).strip()
    
    # 3. Handle hashtag underscores, remove # and @ symbols
    text = re.sub(r'_', ' ', text)
    text = re.sub(r'[@#]', '', text)
    
    # 4. Remove all punctuation EXCEPT:
    # - Word characters & spaces (\w\s)
    # - Exclamation & Question marks (!, ?, Arabic ؟)
    # - Ellipses & periods (. and Unicode …)
    # - Quotation marks (" ' “ ” « »)
    text = re.sub(r'[^\w\s!?؟.…"\'“”«»]', '', text)
    
    # 5. Normalize multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply the function
full['tweet'] = full['tweet'].apply(clean_tweet_punctuation)

full['tweet'].head(10)


0    د. محمود العلايليأرى أن الفريق أحمد شفيق رقم م...
1          مع فيدرر يا آجا والكبار وجه مبتسم بعيون قلب
2    الداعون لمبدأ الاختلاط بين الجنسين كالداعين لإ...
3    ihe 94 ya78m amooo5 badiajnikhar Oukasafa reos...
4    قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حلب...
5    طبيب المقاصة إصابة أحمد الشيخ بسيطة تحتاج للرا...
6    مرسي مش هنام اكتر من اربع ساعات في اليوم.........
7                                  انتخبوا البرص مشكلة
8    Retweeted هاشتاق العرب TheArabHashفيديو أهالي ...
9    مربع صغير أسود محافظ العاصمة تفقد سير العملية ...
Name: tweet, dtype: str

### Translation

In [51]:
from googletrans import Translator
import re 

translator = Translator()

def translate_text(text):
    latin_words = re.findall(r'[a-zA-Z0-9]+', text)

    for la in latin_words:
        if la in text:
            try:
                translated_word = translator.translate(la, src='en', dest='ar').text
                text = re.sub(rf'\b{re.escape(la)}\b', translated_word, text)
            except Exception:
                continue
    return text

full['tweet'] = full['tweet'].apply(translate_text)

C:\Users\pc\AppData\Local\Temp\ipykernel_21472\2925514532.py:15: RuntimeWarning: coroutine 'Translator.translate' was never awaited
  continue


In [52]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,د. محمود العلايليأرى أن الفريق أحمد شفيق رقم م...,False,NEU,msa,0
1,مع فيدرر يا آجا والكبار وجه مبتسم بعيون قلب,False,NEU,msa,1
2,الداعون لمبدأ الاختلاط بين الجنسين كالداعين لإ...,True,NEG,msa,2
3,ihe 94 ya78m amooo5 badiajnikhar Oukasafa reos...,True,NEG,gulf,3
4,قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حلب...,False,NEU,msa,4
5,طبيب المقاصة إصابة أحمد الشيخ بسيطة تحتاج للرا...,False,NEG,msa,5
6,مرسي مش هنام اكتر من اربع ساعات في اليوم.........,True,NEG,egypt,6
7,انتخبوا البرص مشكلة,True,NEG,egypt,7
8,Retweeted هاشتاق العرب TheArabHashفيديو أهالي ...,False,POS,msa,8
9,مربع صغير أسود محافظ العاصمة تفقد سير العملية ...,False,NEU,msa,9


### remove latin untranslated words and trait the repeted words and hhhh


In [53]:
import re

def clean_for_aravec_final(text: str) -> str:
    if not isinstance(text, str):
        return ""
    
    # 1. Strip CSV export quotes from start and end
    text = text.strip()
    text = re.sub(r'^["\'“”«»]+|["\'“”«»]+$', '', text).strip()
    
    # 2. Replace URLs and mentions with a SPACE
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    
    # 3. Replace all hashtags, underscores, and punctuation with a SPACE
    # (This completely stops words like #المال#طرابلس from becoming 'المالطرابلس')
    text = re.sub(r'[#_]', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # 4. Separate numbers from words (e.g. '10دورسينما' -> '10 دورسينما')
    text = re.sub(r'(\d+)([^\W\d_])', r'\1 \2', text)
    text = re.sub(r'([^\W\d_])(\d+)', r'\1 \2', text)
    
    # 5. Remove Tashkeel (Arabic diacritics)
    p_tashkeel = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = re.sub(p_tashkeel, '', text)
    
    # 6. Convert Eastern Arabic numerals to Western digits
    eastern_to_western = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
    text = text.translate(eastern_to_western)
    
    # 7. Normalize Laughter to 'ههه' (length 3)
    text = re.sub(r'ه{2,}', 'ههه', text)
    text = re.sub(r'[hH]{2,}', 'ههه', text)
    
    # 8. AraVec Elongation reduction: reduce 3+ repeated characters to 2
    text = re.sub(r'([^هhH])\1+', r'\1', text)
    text = text.replace('وو', 'و').replace('يي', 'ي').replace('اا', 'ا')
    
    # 9. Official AraVec Letter Normalization (Crucial for Vocab Lookup!)
    # Maps 'أ', 'إ', 'آ' -> 'ا' and 'ة' -> 'ه' and 'ى' -> 'ي'
    search =  ["أ", "إ", "آ", "ة", "ى"]
    replace = ["ا", "ا", "ا", "ه", "ي"]
    for s, r in zip(search, replace):
        text = text.replace(s, r)
        
    # 10. Collapse multiple spaces and trim
    return re.sub(r'\s+', ' ', text).strip()


In [54]:
def delete_latin(text):
    latin_words = re.findall(r'[a-zA-Z0-9]+', text)

    for la in latin_words:
        if la in text:
            text = re.sub(rf'{re.escape(la)}'," ", text)
    return text


In [55]:
full["tweet"] = full["tweet"].apply(delete_latin)
full["tweet"] = full["tweet"].apply(clean_for_aravec_final)

In [56]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,د محمود العلايلياري ان الفريق احمد شفيق رقم مه...,False,NEU,msa,0
1,مع فيدر يا اجا والكبار وجه مبتسم بعيون قلب,False,NEU,msa,1
2,الداعون لمبدا الاختلاط بين الجنسين كالداعين لا...,True,NEG,msa,2
3,مساكين من الصبح و هوما رايحين راجعين عاي غوغل ...,True,NEG,gulf,3
4,قل شرق حلب ولا تقل حلب الشرقيه وقل غرب حلب ولا...,False,NEU,msa,4
5,طبيب المقاصه اصابه احمد الشيخ بسيطه تحتاج لراح...,False,NEG,msa,5
6,مرسي مش هنام اكتر من اربع ساعات في اليوم لو نم...,True,NEG,egypt,6
7,انتخبوا البرص مشكله,True,NEG,egypt,7
8,هاشتاق العرب فيديو اهالي القدس يرفعون الاذان ف...,False,POS,msa,8
9,مربع صغير اسود محافظ العاصمه تفقد سير العمليه ...,False,NEU,msa,9


### Train-Test Duplicate Leakage

In [57]:
full.drop_duplicates(subset=["tweet"], keep=False, inplace=True)

### Data split

In [58]:
train, val = train_test_split(full, test_size=0.10, random_state=42,
stratify=full["sarcasm"])

### Cleaned data export

In [59]:
from pathlib import Path

output_dir = Path("../Data/cleaned_splits/ARAVEC/")
output_file_train = Path(f"{output_dir}\\train-ARAVEC.csv")
output_file_val = Path(f"{output_dir}\\val-ARAVEC.csv")

train.to_csv(output_file_train,index=False)
val.to_csv(output_file_val, index=False)